# 04 — Five detector families, one image, and the score-threshold trap

Run every DOTA detector on the Hub on the same tile, see how differently they spread their confidence scores, and learn why one global score threshold silently hides most of a one-stage detector's output. The lesson ends with RTMDet, the CSPNeXt speed tier and the highest Task 1 score in the DOTA zoo.

| | |
|---|---|
| Needs | Internet (five checkpoints, ≈1.3 GB on first run) |
| Kaggle | Yes — Internet ON, GPU recommended |
| Posts | [Rotated FCOS vs Oriented R-CNN on macOS](https://deeplearning.earth/posts/2026-09-02_rotated_fcos_vs_oriented_rcnn_on_macos/) · [Optical satellite demo](https://deeplearning.earth/posts/2026-09-06_oriented_det_optical_satellite_demo/) · [v0.2.0: Rotated FCOS and decoded rIoU](https://deeplearning.earth/posts/2026-08-28_oriented-det_v0_2_0_rotated_fcos_decoded_riou_and_the_updated_zoo/) |

## Two families of detectors

**Two-stage (R-CNN family).** Stage 1, the region proposal network (RPN), suggests a few thousand candidate regions. Stage 2 crops features for each region (RoIAlign) and classifies and refines it.
- *Oriented R-CNN* proposes **oriented** regions directly and crops them with rotated RoIAlign: the most accurate here.
- *Rotated Faster R-CNN* proposes **horizontal** regions and only predicts the angle in stage 2: simpler and cheaper, slightly less accurate.

**One-stage (dense).** Every location of the feature map predicts a class and a box in one pass.
- *Rotated RetinaNet* places rotated anchors at each location and trains with focal loss: the classic baseline.
- *Rotated FCOS* is anchor-free: each location regresses distances to the four sides, an angle, and a "centerness" score.
- *Rotated RTMDet* is a modern real-time design (CSPNeXt backbone, PAFPN neck, one shared head).

| Hub checkpoint | Official DOTA Task 1 mAP50 | Deploy score floor |
|---|---:|---:|
| `oriented_rcnn_dota_le90_1x` | 76.73% | 0.55 |
| `rotated_faster_rcnn_dota_le90_1x` | 74.42% | 0.60 |
| `rotated_fcos_dota_le90_1x` | 73.07% | 0.20 |
| `rotated_retinanet_dota_le90_1x` | 71.72% | 0.25 |
| `rotated_rtmdet_dota_le90_3x` (3×) | 77.60% | 0.45 |

Rotated RTMDet lands in oriented-det 0.4; until then section 5 needs a checkout of the repo's main branch.

## Scores are not probabilities

A detection score is not a calibrated probability, and each family distributes it differently. Two-stage heads apply a softmax over classes to proposals that already survived filtering, so confident detections pile up near 1. One-stage heads apply a sigmoid focal loss over tens of thousands of locations, and true positives often score between 0.2 and 0.5. OrientedDet therefore stores a per-checkpoint floor in `production.score_threshold`: the best-F1 threshold on validation minus 0.05.

## Setup

The same two cells open every lesson. The first installs OrientedDet: your editable checkout if this notebook sits inside the repo, otherwise the PyPI wheel. The second sets paths and the device, and defines the small helpers used across the series (`hub_files`, `hub_model`, `detect`, `show`, ...). On Kaggle, switch **Internet ON** in the notebook settings.

In [ ]:
# Install: editable checkout when this notebook lives inside the repo, else PyPI.
import importlib.util
from pathlib import Path

REPO = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / "pyproject.toml").is_file() and (p / "oriented_det").is_dir()), None)
_spec = importlib.util.find_spec("oriented_det")
if REPO is None:
    %pip install -q "oriented-det>=0.3.1"
elif _spec is None or not Path(_spec.origin).resolve().is_relative_to(REPO.resolve()):
    %pip install -q -e "{REPO}"

In [ ]:
import os, subprocess, sys, time, urllib.request
from pathlib import Path

import matplotlib.pyplot as plt
import torch
import oriented_det

GH_RAW = "https://raw.githubusercontent.com/DL4EO/oriented-det/main"
IN_KAGGLE = Path("/kaggle/working").is_dir()
HAS_REPO = REPO is not None
WORK = (Path("/kaggle/working") if IN_KAGGLE else
        Path(os.environ.get("ORIENTED_DET_TUTORIAL_WORK", (REPO or Path.cwd()) / "work_dirs")).resolve())
WORK.mkdir(parents=True, exist_ok=True)
os.environ["ORIENTED_DET_PROJECT_ROOT"] = str(WORK)  # runs/ and predictions/ land here
if not HAS_REPO:  # wheel install: keep Hub weights out of site-packages
    os.environ.setdefault("ORIENTED_DET_PRETRAINED_DIR", str(WORK / "pretrained"))

CONFIGS = Path(oriented_det.__file__).parent / "configs"  # recipes shipped with the package
DEVICE = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
ODET = [sys.executable, "-m", "oriented_det.cli"]  # same as typing `odet` in a shell


def run(cmd):
    """Run a CLI command, echoing it first."""
    cmd = [str(c) for c in cmd]
    print("$", " ".join(["odet", *cmd[3:]] if cmd[:3] == ODET else cmd))
    subprocess.run(cmd, check=True)


def demo_file(rel):
    """demo/<rel> from the checkout, else downloaded once from GitHub into WORK/demo."""
    if HAS_REPO and (REPO / "demo" / rel).is_file():
        return REPO / "demo" / rel
    target = WORK / "demo" / rel
    if not target.is_file():
        target.parent.mkdir(parents=True, exist_ok=True)
        urllib.request.urlretrieve(f"{GH_RAW}/demo/{rel}", target)
    return target


def kaggle_input(*needles):
    """First /kaggle/input/<dataset> whose name contains one of `needles` (None off Kaggle)."""
    root = Path("/kaggle/input")
    dirs = sorted(p for p in root.iterdir() if p.is_dir()) if root.is_dir() else []
    return next((p for n in needles for p in dirs if n.lower() in p.name.lower()), None)


def hub_files(slug):
    """Download a Hub checkpoint and its sidecar config once -> (checkpoint_path, config_path)."""
    from huggingface_hub import hf_hub_download
    from oriented_det.pretrained import ensure_checkpoint, load_manifest, resolve_checkpoint_sidecar_config

    ckpt = ensure_checkpoint(f"hf://{slug}")
    manifest = load_manifest()
    sidecar = resolve_checkpoint_sidecar_config(ckpt) or hf_hub_download(
        manifest["repo_id"], ckpt.with_suffix(".json").name,
        revision=manifest.get("revision", "main"), local_dir=ckpt.parent)
    return ckpt, Path(sidecar)


def hub_model(slug, device=DEVICE):
    """Hub checkpoint -> (model, config, class_names). Lesson 01 unpacks it."""
    from oriented_det.runtime.checkpoint import load_model_from_checkpoint

    ckpt, sidecar = hub_files(slug)
    return load_model_from_checkpoint(str(ckpt), str(sidecar), device=device)


def detect(model, config, class_names, image, score_thr=None, classes=None):
    """Detections on a PIL image with the checkpoint's deploy settings (sliding windows past the canvas)."""
    from oriented_det.runtime.inference import apply_nms_to_detections, run_inference_auto
    from oriented_det.train.config import (get_preprocessing_params, resolve_inference_score_threshold,
                                           resolve_inference_sliding_window_overlap_pixels)

    nms = config.production.final_nms_iou_threshold
    dets = run_inference_auto(
        image=image, model=model, device=DEVICE, preprocessing=get_preprocessing_params(config),
        score_threshold=resolve_inference_score_threshold(config) if score_thr is None else score_thr,
        nms_threshold=nms, overlap_pixels=resolve_inference_sliding_window_overlap_pixels(config),
        class_names=class_names)
    dets = apply_nms_to_detections(dets, iou_threshold=nms)
    if classes:
        dets = [d for d in dets if class_names[int(d["label"]) - 1] in classes]
    return dets


def show(*images, titles=(), size=6):
    """Display images side by side."""
    _, axes = plt.subplots(1, len(images), figsize=(size * len(images), size), squeeze=False)
    for ax, img, title in zip(axes[0], images, [*titles, *[""] * len(images)]):
        ax.imshow(img)
        ax.set_title(title)
        ax.axis("off")
    plt.tight_layout()
    plt.show()


print(f"oriented_det: {Path(oriented_det.__file__).parent}")
print(f"device={DEVICE}  kaggle={IN_KAGGLE}  work={WORK}")

## 1. Run all four Hub detectors on the same tile

Keep every candidate down to score 0.05 so the score distributions can be compared. Each model is loaded, timed after one warm-up pass, and freed before the next.

In [ ]:
from PIL import Image

image = Image.open(demo_file("demo.jpg")).convert("RGB")
SLUGS = ["oriented_rcnn_dota_le90_1x", "rotated_faster_rcnn_dota_le90_1x",
         "rotated_fcos_dota_le90_1x", "rotated_retinanet_dota_le90_1x"]
results = {}
for slug in SLUGS:
    model, config, class_names = hub_model(slug)
    if DEVICE != "cpu":
        detect(model, config, class_names, image, score_thr=0.05)  # warm-up
    t0 = time.perf_counter()
    dets = detect(model, config, class_names, image, score_thr=0.05)
    if DEVICE == "cuda":
        torch.cuda.synchronize()
    results[slug] = dict(dets=dets, floor=config.production.score_threshold, seconds=time.perf_counter() - t0,
                         params=sum(p.numel() for p in model.parameters()) / 1e6)
    del model
    if DEVICE == "cuda":
        torch.cuda.empty_cache()
    print(f"{slug:34s} floor {results[slug]['floor']:.2f}   {len(dets):4d} candidates   {results[slug]['seconds']:.2f} s")

## 2. Where do the scores live?

Solid line: the checkpoint's own deploy floor. Dashed line: 0.70, a threshold that "looks reasonable" and that many people apply to every model.

In [ ]:
import numpy as np

fig, axes = plt.subplots(1, 4, figsize=(18, 3.5), sharex=True)
for ax, (slug, r) in zip(axes, results.items()):
    ax.hist([d["score"] for d in r["dets"]], bins=np.linspace(0.05, 1, 39), color="tab:blue")
    ax.axvline(r["floor"], color="k", lw=2)
    ax.axvline(0.70, color="r", ls="--")
    ax.set_yscale("log"); ax.set_title(slug.replace("_dota_le90_1x", ""), fontsize=10); ax.set_xlabel("score")
plt.tight_layout(); plt.show()

## 3. Own floor vs one threshold for all

In [ ]:
from oriented_det.runtime.inference import visualize_results

print(f"{'model':22s} {'kept @ own floor':>17s} {'kept @ 0.70':>12s}")
for slug, r in results.items():
    at_floor = [d for d in r["dets"] if d["score"] >= r["floor"]]
    at_070 = [d for d in r["dets"] if d["score"] >= 0.70]
    r["at_floor"], r["at_070"] = at_floor, at_070
    print(f"{slug.replace('_dota_le90_1x', ''):22s} {len(at_floor):17d} {len(at_070):12d}")

show(*[visualize_results(image.copy(), r["at_floor"], class_names) for r in results.values()],
     titles=[f"{s.replace('_dota_le90_1x', '')} @ {r['floor']}" for s, r in results.items()], size=5)
fcos = results["rotated_fcos_dota_le90_1x"]
show(visualize_results(image.copy(), fcos["at_floor"], class_names),
     visualize_results(image.copy(), fcos["at_070"], class_names),
     titles=(f"FCOS @ its floor {fcos['floor']}: {len(fcos['at_floor'])} boxes",
             f"FCOS @ 0.70: {len(fcos['at_070'])} boxes"))

At 0.70 the two-stage models barely change while FCOS loses most of its correct detections. Comparing models at one shared threshold measures their score calibration, not their accuracy. Use each checkpoint's `production.score_threshold`, or, for mAP, keep everything down to 0.05 (lesson 07).

## 4. Speed, size, and accuracy

In [ ]:
TASK1 = {"oriented_rcnn_dota_le90_1x": 76.73, "rotated_faster_rcnn_dota_le90_1x": 74.42,
         "rotated_fcos_dota_le90_1x": 73.07, "rotated_retinanet_dota_le90_1x": 71.72}
print(f"{'model':22s} {'Task 1 mAP50':>12s} {'params (M)':>11s} {'s / tile (' + DEVICE + ')':>16s}")
for slug, r in results.items():
    print(f"{slug.replace('_dota_le90_1x', ''):22s} {TASK1[slug]:12.2f} {r['params']:11.1f} {r['seconds']:16.2f}")

All four share a ResNet-50 + FPN backbone, so parameter counts are close. The timing differences come from the heads and post-processing (proposal sampling, rotated RoIAlign, NMS over thousands of candidates), not from backbone size.

## 5. RTMDet: the speed tier

Rotated RTMDet swaps ResNet-50 for a CSPNeXt backbone and PAFPN neck, and uses one shared-convolution head with decoded-rIoU box loss. The Hub checkpoint `hf://rotated_rtmdet_dota_le90_3x` comes from `configs/rotated_rtmdet/dota_le90_3x_s.json` (36 epochs, official Task 1 77.60%). This cell prints the network sizes and the recipe, then runs the Hub model on the same tile at its own floor.

In [ ]:
from oriented_det.train.config import TrainingExperimentConfig

try:
    from oriented_det.models.rotated_rtmdet import RotatedRTMDet
except ImportError:
    RotatedRTMDet = None
    print("Rotated RTMDet is not in this oriented-det release (it lands in 0.4); use a checkout of the repo's main branch.")

if RotatedRTMDet is not None:
    for backbone in ("cspnext_tiny", "cspnext_s"):
        net = RotatedRTMDet(num_classes=15, backbone_name=backbone, pretrained_backbone=False)
        print(f"RTMDet-R {backbone:13s} {sum(p.numel() for p in net.parameters()) / 1e6:5.1f} M parameters")
    cfg = TrainingExperimentConfig.load(CONFIGS / "rotated_rtmdet/dota_le90_3x_s.json")
    pre = cfg.preprocessing
    print(f"3x recipe: {cfg.training.num_epochs} epochs, rotate={pre.enable_random_rotate}, "
          f"score_threshold={cfg.production.score_threshold}")

    model, config, class_names = hub_model("rotated_rtmdet_dota_le90_3x")
    if DEVICE != "cpu":
        detect(model, config, class_names, image, score_thr=0.05)  # warm-up
    t0 = time.perf_counter()
    dets = detect(model, config, class_names, image, score_thr=0.05)
    if DEVICE == "cuda":
        torch.cuda.synchronize()
    seconds = time.perf_counter() - t0
    floor = config.production.score_threshold
    at_floor = [d for d in dets if d["score"] >= floor]
    print(f"rotated_rtmdet_dota_le90_3x  Task 1 77.60  {sum(p.numel() for p in model.parameters()) / 1e6:.1f} M params  "
          f"{seconds:.2f} s / tile  {len(at_floor)} boxes @ floor {floor}")
    show(visualize_results(image.copy(), at_floor, class_names),
         titles=(f"RTMDet-S 3x @ {floor}: {len(at_floor)} boxes",))
    del model

## What to try next

- Swap the 1× slugs for the 3× ones (`..._3x`). Do the score distributions shift?
- Pick the threshold for FCOS yourself: count detections in 0.05 steps and look for the knee.
- Run the zoo on `demo/large.jpg`. `detect()` switches to sliding windows automatically; lesson 06 explains how.
- Train an RTMDet recipe with `odet train --config <recipe>` (lesson 07 covers recipes and patching).
- **Next:** [05 — The fifteen DOTA classes](05_dota_fifteen_classes.ipynb).